# **RoboPianist drum performance tutorial**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/robopianist/blob/main/drum_tutorial.ipynb)


> <p><small><small>Copyright 2024 The RoboPianist Authors.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>


# Installing RoboPianist and drum demo dependencies


In [1]:
# @title Install dependencies (Colab users: enable a GPU runtime first)
from IPython.display import clear_output

print("Installing packages...")
%pip install -q robopianist>=1.0.6 dm-control>=1.0.16 imageio[ffmpeg]
%env MUJOCO_GL=egl

clear_output()
print("Dependencies installed. If you are running on Colab, remember to enable a GPU runtime.")


Dependencies installed. If you are running on Colab, remember to enable a GPU runtime.


# Imports


In [2]:
# @title Imports used throughout the notebook
from base64 import b64encode
from copy import deepcopy
from pathlib import Path

import os
import shutil
import subprocess
import wave

os.environ.setdefault('MUJOCO_GL', 'egl')

import imageio.v2 as imageio
import numpy as np
import fluidsynth
from IPython.display import HTML, Audio, display

from dm_control import mjcf
from robopianist import SF2_PATH
from robopianist.models.drum import drum
from robopianist.music import midi_message


# Helper functions


In [3]:
# @title Utility helpers for visualization, simulation, and audio

VIDEO_DIR = Path('videos')
VIDEO_DIR.mkdir(exist_ok=True)

def play_video(filename: str) -> None:
    """Embed an MP4 video directly inside the notebook."""
    path = Path(filename)
    with path.open('rb') as f:
        video = f.read()
    b64 = b64encode(video).decode('utf-8')
    display(
        HTML(
            f"""
            <video width="720" controls loop>
              <source src="data:video/mp4;base64,{b64}" type="video/mp4">
            </video>
            """
        )
    )


def interpolate_controls(times: np.ndarray, values: np.ndarray, t: float) -> np.ndarray:
    """Linearly interpolate joint targets at time ``t``."""
    return np.array([np.interp(t, times, values[:, i]) for i in range(values.shape[1])])


def simulate_and_render(
    physics: mjcf.Physics,
    times: np.ndarray,
    values: np.ndarray,
    *,
    drum_entity=None,
    random_state=None,
    camera_id: str = 'front',
    fps: int = 30,
    hold_steps: int = 45,
    resolution: tuple[int, int] = (480, 640),
) -> list[np.ndarray]:
    """Run the control sequence and return RGB frames."""
    frames: list[np.ndarray] = []
    dt = physics.timestep()
    steps_per_frame = max(1, int(round((1.0 / fps) / dt)))
    total_steps = int(np.ceil(times[-1] / dt))
    rng = random_state or np.random.RandomState()

    for step in range(total_steps):
        t = step * dt
        ctrl = interpolate_controls(times, values, t)
        physics.data.ctrl[:] = ctrl
        physics.step()
        if drum_entity is not None:
            drum_entity.after_substep(physics, rng)
        if step % steps_per_frame == 0:
            frame = physics.render(height=resolution[0], width=resolution[1], camera_id=camera_id)
            frames.append(frame)

    for _ in range(hold_steps):
        physics.data.ctrl[:] = values[-1]
        physics.step()
        if drum_entity is not None:
            drum_entity.after_substep(physics, rng)
        frame = physics.render(height=resolution[0], width=resolution[1], camera_id=camera_id)
        frames.append(frame)

    return frames


def synthesize_drum_audio(
    events: list[midi_message.MidiMessage],
    *,
    sample_rate: int = 44100,
    soundfont_path: Path = SF2_PATH,
    trailing_silence: float = 0.8,
) -> np.ndarray:
    """Render drum MIDI events with FluidSynth."""
    if not events:
        return np.zeros(0, dtype=np.int16)

    ordered = sorted(deepcopy(events), key=lambda e: e.time)
    current_time = ordered[0].time
    for idx in range(len(ordered) - 1):
        delta = max(0.0, ordered[idx + 1].time - ordered[idx].time)
        ordered[idx].time = delta
    ordered[-1].time = max(trailing_silence, 0.0)

    synth = fluidsynth.Synth(samplerate=float(sample_rate))
    soundfont_id = synth.sfload(str(soundfont_path))
    synth.program_select(9, soundfont_id, 128, 0)

    total_duration = current_time + sum(event.time for event in ordered)
    total_samples = int(np.ceil(sample_rate * total_duration))
    total_samples = max(total_samples, sample_rate // 2)
    buffer = np.zeros(total_samples, dtype=np.float32)
    sample_cursor = int(round(current_time * sample_rate))

    for event in ordered:
        if isinstance(event, midi_message.NoteOn):
            synth.noteon(9, event.note, event.velocity)
        elif isinstance(event, midi_message.NoteOff):
            synth.noteoff(9, event.note)
        elif isinstance(event, midi_message.SustainOn):
            synth.cc(9, 64, 127)
        elif isinstance(event, midi_message.SustainOff):
            synth.cc(9, 64, 0)
        else:
            raise ValueError(f'Unsupported MIDI event type: {event}')

        duration = max(0.0, event.time)
        sample_count = int(round(duration * sample_rate))
        if sample_count > 0:
            chunk = np.asarray(synth.get_samples(sample_count))[::2]
            end_sample = sample_cursor + chunk.shape[0]
            if end_sample > buffer.shape[0]:
                buffer = np.pad(buffer, (0, end_sample - buffer.shape[0]))
            buffer[sample_cursor:end_sample] += chunk
            sample_cursor = end_sample
        else:
            sample_cursor += sample_count

    synth.all_notes_off(9)
    synth.delete()

    peak = np.max(np.abs(buffer))
    if peak > 0:
        buffer /= peak
    return (buffer * np.iinfo(np.int16).max).astype(np.int16)


def write_wave(path: Path, samples: np.ndarray, sample_rate: int = 44100) -> None:
    path.parent.mkdir(exist_ok=True)
    with wave.open(str(path), 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(samples.tobytes())


def add_audio_to_video(video_path: Path, audio_path: Path, output_path: Path) -> None:
    video_path = Path(video_path)
    audio_path = Path(audio_path)
    output_path = Path(output_path)

    if output_path == video_path:
        temp_video = video_path.with_suffix('.temp.mp4')
        shutil.copyfile(video_path, temp_video)
        input_video = temp_video
    else:
        input_video = video_path

    try:
        subprocess.run(
            [
                'ffmpeg',
                '-nostdin',
                '-y',
                '-i',
                str(input_video),
                '-i',
                str(audio_path),
                '-c:v',
                'copy',
                '-c:a',
                'aac',
                '-shortest',
                str(output_path),
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.STDOUT,
        )
    finally:
        if output_path == video_path:
            Path(input_video).unlink(missing_ok=True)


# Drum kit with a simple robotic arm

We instantiate the `robopianist.models.drum.Drum` composer entity (which already tracks strike velocities and emits MIDI) and attach a lightweight 4-DoF striker arm so it can reach the kick drum and ride cymbal. The helper returns joint metadata that we reuse for trajectory planning.


In [4]:
# @title Build a procedural drum kit with a 4-DoF striking arm
import numpy as np

def add_xarm7_style_striker(
    parent: mjcf.Element,
    name: str = "xarm7_striker",
    base_pos=( -0.20, -0.55, 0.35),     # 机械臂基座安装位置（世界系/父系）
    scale: float = 1.0,                 # 尺寸整体缩放（默认按 xArm7 实尺）
    stick_length: float = 0.35,         # 鼓槌长度（沿 +X 伸出）
    add_actuators: bool = True,
    kp: float = 250.0,                  # 位置伺服刚度
):
    """
    在 parent 下添加一个 4-DoF 的击打机械臂（Z-Y-Y-Z），尺寸/关节范围参考 xArm7。
    返回: dict，包含 joints, joint_names, ee_site, root_body
    """

    # ---- xArm7 近似尺寸（单位 m），可被 scale 缩放 ----
    d1 = 0.267 * scale     # 基座抬高（xArm7 d1）
    L_upper = 0.293 * scale   # 上臂近似（参考 xArm7 d3）
    L_fore  = 0.3425 * scale  # 前臂近似（参考 xArm7 d5）
    L_wrist = 0.097 * scale   # 腕段近似（参考 xArm7 d7）

    # 杆件半径（仅外观/碰撞）
    r_base   = 0.05  * np.sqrt(scale)
    r_upper  = 0.04  * np.sqrt(scale)
    r_fore   = 0.035 * np.sqrt(scale)
    r_wrist  = 0.025 * np.sqrt(scale)
    r_stick  = 0.010

    # ---- 关节范围（弧度），参考 xArm7，做了对称/稳定性微调 ----
    # j1_range = (-np.pi, np.pi)              # 可放宽到 (-2*np.pi, 2*np.pi)
    j1_range = (-2*np.pi, 2*np.pi)
    j2_range = (np.deg2rad(-118), np.deg2rad(120))
    j3_range = (np.deg2rad(-11),  np.deg2rad(225))
    j4_range = (-np.pi, np.pi)

    # ---- 基座与立柱（固定）----
    arm_mount = parent.add('body', name=f'{name}_mount', pos=base_pos)
    arm_mount.add('geom', type='capsule',
                  fromto=[0, 0, -0.35*scale, 0, 0, 0.05*scale],
                  size=[max(0.06*np.sqrt(scale), 0.03)],
                  rgba=[0.20, 0.20, 0.20, 1.0])

    # 肩部基座（无关节，仅外观）
    shoulder_base = arm_mount.add('body', name=f'{name}_shoulder_base', pos=[0, 0, 0.05*scale])
    shoulder_base.add('geom', type='capsule',
                      fromto=[0,0,0, 0,0,d1],
                      size=[r_base], rgba=[0.30,0.30,0.35,1.0])

    # ---- J1: base yaw (Z) ----
    j1_body = shoulder_base.add('body', name=f'{name}_j1_body', pos=[0,0,d1])
    # 给 j1_body 添加惯量，避免“关节 body 无质量/无惯性”报错
    j1_mass = 0.3                         # kg
    j1_radius = 0.05 * scale              # m，假想等效球半径
    I = 2.0/5.0 * j1_mass * (j1_radius**2)  # 球的转动惯量
    j1_body.add('inertial', pos=[0, 0, 0], mass=j1_mass, diaginertia=[I, I, I])

    j1 = j1_body.add('joint', name=f'{name}_j1_yaw', type='hinge',
                     axis=[0,0,1], limited=True, range=j1_range, damping=2.5)

    # ---- J2: shoulder pitch (Y) + 上臂 ----
    upper_arm = j1_body.add('body', name=f'{name}_upper', pos=[0,0,0.0])
    j2 = upper_arm.add('joint', name=f'{name}_j2_pitch', type='hinge',
                       axis=[0,1,0], limited=True, range=j2_range, damping=1.5)
    upper_arm.add('geom', type='capsule',
                  fromto=[0,0,0, 0,0,L_upper],
                  size=[r_upper], rgba=[0.40,0.40,0.45,1.0])

    # ---- J3: elbow pitch (Y) + 前臂 ----
    forearm = upper_arm.add('body', name=f'{name}_fore', pos=[0,0,L_upper])
    j3 = forearm.add('joint', name=f'{name}_j3_pitch', type='hinge',
                     axis=[0,1,0], limited=True, range=j3_range, damping=1.2)
    forearm.add('geom', type='capsule',
                fromto=[0,0,0, 0,0,L_fore],
                size=[r_fore], rgba=[0.50,0.50,0.55,1.0])

    # ---- J4: wrist yaw (Z) + 腕段 ----
    wrist = forearm.add('body', name=f'{name}_wrist', pos=[0,0,L_fore])
    j4 = wrist.add('joint', name=f'{name}_j4_yaw', type='hinge',
                   axis=[0,0,1], limited=True, range=j4_range, damping=0.6)
    wrist.add('geom', type='capsule',
              fromto=[0,0,0, 0,0,L_wrist],
              size=[r_wrist], rgba=[0.45,0.45,0.50,1.0])

    # ---- 鼓槌（沿 +X 方向伸出）+ 末端 site ----
    stick = wrist.add('body', name=f'{name}_stick', pos=[0,0,L_wrist])
    stick.add('geom', name=f'{name}_stick_geom', type='capsule',
              fromto=[0,0,0, stick_length,0,0],
              size=[r_stick], rgba=[0.80,0.60,0.30,1.0])
    ee = stick.add('site', name=f'{name}_tip', pos=[stick_length,0,0],
                   size=[0.01], rgba=[1,0,0,1])

    joints = [j1, j2, j3, j4]
    joint_names = [j.name for j in joints]

    if add_actuators:
        for j in joints:
            parent.root.actuator.add('position',
                                     name=f'{j.name}_act',
                                     joint=j,
                                     ctrlrange=list(j.range),
                                     kp=kp)

    return {
        'base_pos':base_pos,
        'd1':d1,'L_upper':L_upper,'L_fore':L_fore,'L_wrist':L_wrist,
        'stick_length':stick_length,
        'j1_range':j1_range,
        'j2_range':j2_range,
        'j3_range':j3_range,
        'j4_range':j4_range,
        'joint_names':joint_names,
        
    }

drum_entity = drum.Drum(add_actuators=False)
drum_model = drum_entity.mjcf_model
drum_model.option.timestep = 0.002
world = drum_model.worldbody

arm = add_xarm7_style_striker(
    parent=world,
    base_pos=(1.20, 0.2, 0.35),
    scale=1.0,          # 整体放大/缩小
    stick_length=0.35,
    add_actuators=True,
    kp=250
)

physics = mjcf.Physics.from_mjcf_model(drum_model)
physics.forward()

drum_random_state = np.random.RandomState(0)
drum_entity.initialize_episode(physics, drum_random_state)


# Define a striking motion

Plan a minimum-jerk strike that hits the kick drum and the ride cymbal in sequence, pausing briefly at impact before returning to the ready pose.


In [5]:
# @title Keyframe targets for a snare hit

from DLS_demo import plan_strike_trajectory

time_duration = 1.5   
holdon_time = 0.1
init_q = np.array([-0.3 - np.pi, 0.4, 0.3, 0.0])   
# init_q = np.array([-0.3 - np.pi, 0.4, 0.6, 0.0])   
base_pos = arm['base_pos']
# target_pos = physics.named.data.site_xpos[f'kick_strike_site'].copy()
target_pos = physics.named.data.site_xpos[f'ride_strike_site'].copy()

# plan the strike trajectory
keyframe_times1, keyframe_values1 = plan_strike_trajectory(
    base_pos,init_q,target_pos,time_duration,arm
)

# hold on for 0.5s at the target position
keyframe_times2 = keyframe_times1[-1] + np.array([holdon_time / 2, holdon_time])
keyframe_values2 = np.tile(keyframe_values1[-1], (2, 1))

# back to initial pose
keyframe_times3 = keyframe_times2[-1] + keyframe_times1[1:]
keyframe_values3 = keyframe_values1[-1:0:-1]

# concatenate all keyframes
keyframe_times = np.concatenate([keyframe_times1, keyframe_times2, keyframe_times3])
keyframe_values = np.vstack([keyframe_values1, keyframe_values2, keyframe_values3])

# # add another strike to the ride cymbal
# target_pos = physics.named.data.site_xpos[f'ride_strike_site'].copy()
# print("Target pos:", target_pos)
# keyframe_times1, keyframe_values1 = plan_strike_trajectory(
#     base_pos,init_q,target_pos,time_duration,arm
# )
# keyframe_times1 += keyframe_times[-1]  # shift time


# keyframe_times2 = keyframe_times1[-1] + np.array([holdon_time / 2, holdon_time])
# keyframe_values2 = np.tile(keyframe_values1[-1], (2, 1))

# keyframe_times3 = keyframe_times2[-1] + keyframe_times1[1:]
# keyframe_values3 = keyframe_values1[-1:0:-1]

# # concatenate all keyframes
# keyframe_times = np.concatenate([keyframe_times, keyframe_times1, keyframe_times2, keyframe_times3])
# keyframe_values = np.vstack([keyframe_values, keyframe_values1, keyframe_values2, keyframe_values3])


In [6]:
import numpy as np

def plan_macro_keyframes(
    physics,
    arm_config,                    # 你的 arm 配置字典（含 d1/L_upper/L_fore/L_wrist/stick_length & j*_range）
    base_pos,                      # 机械臂基座 (x,y,z)
    q1, q2,                        # 两个已知的稳定初姿态（np.array shape (4,)）
    target: str,                   # 'ride' 或 'kick'
    time_to_target: float = 1.5,   # q→target 的主段时间
    dwell: float = 0.1,            # 目标处停留
    pre_move_time: float = 0.5,    # q1<->q2 预移动时间（仅 kick 用）
    ride_site_name: str = 'ride_strike_site',
    kick_site_name: str = 'kick_strike_site',
    n_main: int = 11,              # 主段关键帧数
    n_pre: int = 11,               # 预移动关键帧数
):
    """
    返回:
      keyframe_times: 1D np.ndarray，严格递增
      keyframe_values: 2D np.ndarray [N,4]，每行是4个关节角(rad)
    依赖你已有的 plan_strike_trajectory(base_pos, init_q, target_pos, time, arm_config)
    """

    def min_jerk_q(q_start, q_end, T, n):
        t = np.linspace(0.0, T, n)
        if T <= 0:
            Q = np.tile(q_start, (n,1))
        else:
            s = t / T
            S = 10*s**3 - 15*s**4 + 6*s**5
            Q = q_start[None,:] + (q_end - q_start)[None,:] * S[:,None]
        return t, Q

    # 查目标位姿
    if target == 'ride':
        target_pos = physics.named.data.site_xpos[ride_site_name].copy()
    elif target == 'kick':
        target_pos = physics.named.data.site_xpos[kick_site_name].copy()
    else:
        raise ValueError("target must be 'ride' or 'kick'.")

    # ---- case 1: ride ----
    if target == 'ride':
        # q1 -> ride
        t1, Q1 = plan_strike_trajectory(base_pos, q1, target_pos, time_to_target, arm_config)
        t1 = np.array(t1); Q1 = np.array(Q1)

        # 停留
        t2 = t1[-1] + np.array([dwell/2, dwell])
        Q2 = np.tile(Q1[-1], (2,1))

        # 原路返回（时间对齐 + 反转）
        t3 = t2[-1] + t1[1:]
        Q3 = Q1[-1:0:-1]

        times = np.concatenate([t1, t2, t3])
        values = np.vstack([Q1, Q2, Q3])
        return times, values

    # ---- case 2: kick ----
    else:
        # 预移动：q1 -> q2（关节空间）
        t0, Q0 = min_jerk_q(q1, q2, pre_move_time, n_pre)
        t0 = np.array(t0); Q0 = np.array(Q0)

        # q2 -> kick（IK）
        t1, Q1 = plan_strike_trajectory(base_pos, q2, target_pos, time_to_target, arm_config)
        t1 = np.array(t1) + t0[-1]                  # 时间偏移到衔接
        Q1 = np.array(Q1)

        # 停留
        t2 = t1[-1] + np.array([dwell/2, dwell])
        Q2 = np.tile(Q1[-1], (2,1))

        # 原路返回（kick段反向）
        t3 = t2[-1] + (t1 - t1[0])[1:]             # 使用与去程相同的相对时间
        Q3 = Q1[-1:0:-1]

        # 收尾：q2 -> q1（关节空间）
        t4, Q4 = min_jerk_q(q2, q1, pre_move_time, n_pre)
        t4 = t3[-1] + t4
        Q4 = np.array(Q4)

        times = np.concatenate([t0, t1, t2, t3, t4])
        values = np.vstack([Q0, Q1, Q2, Q3, Q4])
        return times, values

q1 = np.array([-0.3 - np.pi, 0.4, 0.3, 0.0])
q2 = np.array([-0.3 - np.pi, 0.4, 0.6, 0.0])
keyframe_times1,keyframe_values1 = plan_macro_keyframes(
    physics,
    arm,
    base_pos,
    q1,q2,
    target='kick',
    time_to_target=time_duration,
    dwell=holdon_time
)

keyframe_times2,keyframe_values2 = plan_macro_keyframes(
    physics,
    arm,
    base_pos,
    q1,q2,
    target='ride',
    time_to_target=time_duration,
    dwell=holdon_time
)

keyframe_times2 = keyframe_times1[-1] + keyframe_times2[1:]
keyframe_values2 = keyframe_values2[1:]

keyframe_times = np.concatenate([keyframe_times1, keyframe_times2])
keyframe_values = np.vstack([keyframe_values1, keyframe_values2])

# Simulate and render the drum performance


In [7]:
# @title Run the trajectory and export a video

drum_random_state = np.random.RandomState(1)
drum_entity.initialize_episode(physics, drum_random_state)

for joint_name, value in zip(arm["joint_names"], keyframe_values[0]):
    physics.named.data.qpos[joint_name] = value
physics.forward()

frames = simulate_and_render(
    physics,
    times=keyframe_times,
    values=keyframe_values,
    drum_entity=drum_entity,
    random_state=drum_random_state,
    camera_id='front',
    fps=30,
    hold_steps=60,
    resolution=(480, 640),
)

output_path = VIDEO_DIR / 'drum_snare_hit.mp4'
imageio.mimsave(output_path, frames, fps=30, macro_block_size=None)

print(f'Saved video to {output_path}')
video_duration = len(frames) / 30.0
print(f'Captured {len(frames)} frames at 30 FPS ({video_duration:.2f}s)')

drum_midi_events = drum_entity.midi_module.get_all_midi_messages()
note_on_events = [event for event in drum_midi_events if isinstance(event, midi_message.NoteOn)]
print(f'Collected {len(drum_midi_events)} MIDI events ({len(note_on_events)} note-on messages)')
if note_on_events:
    unique_notes = sorted({event.note for event in note_on_events})
    print(f'Unique percussion notes: {unique_notes}')

play_video(output_path)


Saved video to videos/drum_snare_hit.mp4
Captured 272 frames at 30 FPS (9.07s)
Collected 0 MIDI events (0 note-on messages)


# Add FluidSynth drum audio

We feed the recorded MIDI stream into FluidSynth on the General MIDI percussion channel and mux the rendered waveform into the exported video.


In [8]:
# @title Generate drum audio from recorded MIDI and attach it to the video

audio_samples = synthesize_drum_audio(drum_midi_events, sample_rate=44100)
if audio_samples.size == 0 or not np.any(audio_samples):
    raise RuntimeError('Generated audio is silent; re-run the trajectory or adjust the strike plan.')

audio_path = VIDEO_DIR / 'drum_snare_hit.wav'
write_wave(audio_path, audio_samples)

muxed_video_path = VIDEO_DIR / 'drum_snare_hit_with_audio.mp4'
add_audio_to_video(output_path, audio_path, muxed_video_path)

peak = int(np.max(np.abs(audio_samples)))
print(f'Saved audio track to {audio_path}')
print(f'Updated video with audio at {muxed_video_path}')
print(f'Audio peak amplitude: {peak}')

display(Audio(audio_samples, rate=44100))
play_video(muxed_video_path)


RuntimeError: Generated audio is silent; re-run the trajectory or adjust the strike plan.